# Spark trên dữ liệu thật: NYC Yellow Taxi 2019

Bộ `nyc-tlc` là **mọi chuyến Yellow Taxi ở New York năm 2019** do NYC Taxi & Limousine Commission
công bố: 12 tệp Parquet (~1,2 GB, **~84,6 triệu chuyến**) cùng bảng `taxi_zone_lookup.csv`
(265 khu vực đón/trả khách). Quá lớn để pandas đọc thoải mái trên laptop, vừa đủ để thấy Spark khác gì.

Việc của bạn, theo đúng thứ tự một job batch thật:

1. **load**: đọc 12 tháng, chỉ lấy cột cần, đặt tên và kiểu thống nhất;
2. **clean**: gắn lý do loại cho từng chuyến hỏng (thời gian âm, quãng đường 0, giá âm...), giữ chuyến tốt;
3. **write**: ghi Parquet **partition theo ngày đón khách** vào `data/clean/nyc-tlc/yellow_2019/`,
   mỗi ngày một tệp, chạy lại không nhân đôi;
4. **analyze**: giờ cao điểm, tip theo cách trả tiền, tip theo borough (join bảng zone bằng **broadcast**), sân bay;
5. **tối ưu**: đo thời gian trước/sau (broadcast vs sort-merge join, small files, partition pruning);
6. vẽ biểu đồ, rồi làm phần trắc nghiệm trong app.

Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, đo giờ và vẽ; khi chấm chúng bị bỏ
qua, test gọi thẳng các hàm bạn viết trên DataFrame nhỏ và trên bộ thật. Chấm:
`uv run pytest tracks/data-engineer/modules/06-spark-big-data/exercises/intermediate/04-nyc-taxi-real -q`

Chạy cả notebook mất khoảng 1,5–2 phút trên máy 16 nhân (12 tháng, không cần chọn tháng con).

In [ ]:
# explore: tìm gốc repo (thư mục có pyproject.toml), rồi tải bộ dữ liệu nếu chưa có (~1,2 GB)
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

RAW = ensure("nyc-tlc")
CLEAN = ROOT / "data" / "clean" / "nyc-tlc" / "yellow_2019"
MONTHS = sorted(RAW.glob("yellow_tripdata_2019-*.parquet"))
print(RAW, len(MONTHS), "tháng", f"{sum(p.stat().st_size for p in MONTHS) / 1e9:.2f} GB")

In [ ]:
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

# Cột lấy từ tệp TLC -> (tên mới, kiểu). Bỏ các cột phụ phí không dùng tới.
COLUMNS = {
    "tpep_pickup_datetime": ("pickup_at", "timestamp_ntz"),
    "tpep_dropoff_datetime": ("dropoff_at", "timestamp_ntz"),
    "passenger_count": ("passenger_count", "int"),
    "trip_distance": ("trip_distance", "double"),
    "PULocationID": ("pu_location_id", "int"),
    "DOLocationID": ("do_location_id", "int"),
    "payment_type": ("payment_type", "int"),
    "fare_amount": ("fare_amount", "double"),
    "tip_amount": ("tip_amount", "double"),
    "total_amount": ("total_amount", "double"),
}
START, END = "2019-01-01", "2020-01-01"     # chỉ giữ chuyến đón khách trong năm 2019: [START, END)
MAX_MINUTES, MAX_MILES, MAX_FARE, MAX_MPH = 180, 100, 500, 80
REASONS = ["out_of_period", "bad_duration", "bad_distance", "bad_amount", "too_fast"]  # thứ tự ưu tiên
PAYMENT = {1: "card", 2: "cash"}            # payment_type khác (no charge, dispute, 0...) -> "other"
AIRPORT_ZONES = ("Airports", "EWR")         # service_zone của JFK, LaGuardia và Newark


def load_zones(spark: SparkSession, path) -> DataFrame:
    """taxi_zone_lookup.csv -> location_id, borough, zone, service_zone. Đã viết sẵn."""
    raw = spark.read.csv(str(path), header=True)
    return raw.select(
        F.col("LocationID").cast("int").alias("location_id"),
        F.col("Borough").alias("borough"),
        F.col("Zone").alias("zone"),
        F.col("service_zone"),
    )


def duration_minutes() -> F.Column:
    """Số phút từ pickup_at tới dropoff_at. Đã viết sẵn.

    Cột là TIMESTAMP_NTZ (giờ đồng hồ ở New York, không kèm múi giờ) nên không `cast("long")` được;
    `timestampdiff` trừ thẳng hai mốc, không phụ thuộc múi giờ của session.
    """
    return F.expr("timestampdiff(SECOND, pickup_at, dropoff_at)") / 60


def tip_rate() -> F.Column:
    """Cột tip_rate = sum(tip_amount) / sum(fare_amount), làm tròn 4. Đã viết sẵn, dùng trong .agg(...)."""
    return F.round(F.sum("tip_amount") / F.sum("fare_amount"), 4).alias("tip_rate")

In [ ]:
# explore: SparkSession local dùng hết nhân; driver.memory chỉ có tác dụng khi JVM chưa chạy
import tempfile
import time

spark = (
    SparkSession.builder.master("local[*]")
    .appName("nyc-taxi-2019")
    .config("spark.driver.memory", "8g")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.warehouse.dir", tempfile.mkdtemp(prefix="spark-wh-"))
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
zones = load_zones(spark, RAW / "taxi_zone_lookup.csv")
print(spark.version, spark.sparkContext.defaultParallelism, "nhân,", zones.count(), "zone")

## 1. Đọc dữ liệu

`load_trips(spark, paths)` đọc các tệp Parquet trong `paths` (danh sách `Path`) **một lần** (`spark.read.parquet(*paths)`),
chỉ giữ các cột trong `COLUMNS`, đổi tên và ép kiểu theo đúng thứ tự khai báo. Kết quả có 10 cột:
`pickup_at, dropoff_at, passenger_count, trip_distance, pu_location_id, do_location_id, payment_type,
fare_amount, tip_amount, total_amount`.

Parquet là columnar: `select` sớm nghĩa là Spark chỉ đọc 10/19 cột từ đĩa (**column pruning**).

In [ ]:
def load_trips(spark: SparkSession, paths) -> DataFrame:
    # TODO: spark.read.parquet(*[str(p) for p in paths]), rồi select từng cột trong COLUMNS:
    #       F.col(tên_gốc).cast(kiểu).alias(tên_mới)
    raise NotImplementedError

In [ ]:
# explore: 12 tệp, mỗi tệp chỉ 1 row group -> mỗi tệp đọc bằng 1 task, dù tệp lớn hơn 128 MB
raw = load_trips(spark, MONTHS)
raw.printSchema()
t = time.perf_counter()
RAW_ROWS = raw.count()
print(f"{RAW_ROWS:,} chuyến, {raw.rdd.getNumPartitions()} partition đầu vào, đếm mất {time.perf_counter() - t:.1f}s")
# count() không lọc gì nên Spark đọc số dòng trong footer Parquet, không đọc dữ liệu: nhanh bất thường

## 2. Làm sạch

Dữ liệu TLC do máy tính tiền trên xe gửi về, nhiều chuyến vô lý. Mỗi chuyến hỏng nhận **một** lý do:
lý do đứng trước trong `REASONS` thắng khi vi phạm nhiều luật. Giá trị `null` ở các cột trong luật
(vd. thiếu `dropoff_at`) tính là vi phạm.

| Lý do | Chuyến bị loại khi |
| --- | --- |
| `out_of_period` | `pickup_at` không nằm trong `[START, END)` (tệp 2019 có chuyến năm 2001, 2008, 2088...) |
| `bad_duration` | thời gian chuyến (phút, `duration_minutes()`) `<= 0` hoặc `> MAX_MINUTES` |
| `bad_distance` | `trip_distance <= 0` hoặc `> MAX_MILES` (dặm) |
| `bad_amount` | `fare_amount <= 0` hoặc `> MAX_FARE`, hoặc `tip_amount < 0`, hoặc `total_amount <= 0` |
| `too_fast` | tốc độ trung bình `trip_distance / (phút / 60)` lớn hơn `MAX_MPH` dặm/giờ |

- `flag_rejects(df)`: thêm cột `reject_reason` (chuỗi, `null` nếu chuyến tốt). Giữ nguyên các cột cũ.
- `clean_trips(df)`: chỉ giữ chuyến có `reject_reason` là `null`, bỏ cột đó, rồi thêm theo thứ tự
  `duration_min` (làm tròn 2), `pickup_hour` (0–23) và `pickup_date` (kiểu `date`) tính từ `pickup_at`.
  Thiếu `passenger_count` **không** phải lý do loại: tài xế hay bỏ trống, mà ta không dùng cột này.

Không dedup: chuyến taxi không có khoá; 50 dòng trùng hệt nhau trên 84,6 triệu không đáng một lần shuffle cả bộ.

In [ ]:
def flag_rejects(df: DataFrame) -> DataFrame:
    # TODO: với mỗi luật, viết điều kiện "chuyến tốt" (Column boolean), rồi dựng cột reject_reason
    #       bằng F.when(...).otherwise(...) lồng nhau sao cho luật đứng trước trong REASONS thắng.
    #       Điều kiện ra null (thiếu dữ liệu) phải tính là vi phạm: F.coalesce(dieu_kien, F.lit(False)).
    raise NotImplementedError


def clean_trips(df: DataFrame) -> DataFrame:
    # TODO: flag_rejects -> giữ reject_reason null -> drop -> thêm duration_min, pickup_hour, pickup_date
    raise NotImplementedError

In [ ]:
# explore: bao nhiêu chuyến bị loại, vì sao (một lần quét cả 84,6 triệu dòng)
import pandas as pd

t = time.perf_counter()
reasons = flag_rejects(raw).groupBy("reject_reason").count().toPandas()
reasons["reject_reason"] = reasons["reject_reason"].fillna("ok")
reasons = reasons.set_index("reject_reason")["count"].sort_values(ascending=False)
REJECTED = int(reasons.drop("ok").sum())
print(f"quét mất {time.perf_counter() - t:.1f}s; loại {REJECTED:,} chuyến = {REJECTED / RAW_ROWS:.2%}")
print(reasons.to_frame("trips").assign(pct=lambda d: (d["trips"] / RAW_ROWS * 100).round(3)))

In [ ]:
# explore: biểu đồ lý do loại
import matplotlib.pyplot as plt

ax = reasons.drop("ok").sort_values().plot.barh(figsize=(8, 3), color="tab:red")
ax.set_title(f"Chuyến bị loại theo lý do ({REJECTED:,} / {RAW_ROWS:,} chuyến)")
ax.set_xlabel("chuyến")
plt.show()

## 3. Ghi dữ liệu sạch, partition theo ngày

`write_clean(df, out_dir)` ghi `df` thành Parquet ở `out_dir`, **partition theo `pickup_date`**
(thư mục `pickup_date=2019-07-04/`), **ghi đè** (`mode("overwrite")`) để chạy lại không nhân đôi, và trả
`str(out_dir)`.

Mỗi task Spark ghi một tệp vào **mỗi** thư mục ngày mà nó có dữ liệu. Nếu dữ liệu đang rải đều 64
partition (sau một shuffle) thì mỗi ngày nhận 64 tệp nhỏ, cả năm ~23 nghìn tệp: **small file problem**
(đọc chậm, NameNode/S3 list chậm). Yêu cầu: **mỗi ngày đúng một tệp**. Gợi ý: `repartition` theo chính
cột partition trước khi ghi, để mọi dòng của một ngày về cùng một task.

In [ ]:
def write_clean(df: DataFrame, out_dir) -> str:
    # TODO: repartition theo pickup_date, rồi write.mode("overwrite").partitionBy("pickup_date").parquet(...)
    raise NotImplementedError

In [ ]:
# explore: ghi cả năm vào data/clean/nyc-tlc/yellow_2019/, đếm thư mục ngày và tệp
t = time.perf_counter()
write_clean(clean_trips(raw), CLEAN)
print(f"ghi mất {time.perf_counter() - t:.1f}s")
day_dirs = sorted(CLEAN.glob("pickup_date=*"))
files = list(CLEAN.glob("pickup_date=*/*.parquet"))
size_gb = sum(f.stat().st_size for f in files) / 1e9
print(f"{len(day_dirs)} thư mục ngày, {len(files)} tệp Parquet, {size_gb:.2f} GB")

# Từ đây phân tích đọc lại dữ liệu sạch (như job downstream thật), không làm sạch lại từ đầu
trips = spark.read.parquet(str(CLEAN))
CLEAN_ROWS = trips.count()
print(f"{CLEAN_ROWS:,} chuyến sạch")

## 4. Phân tích

Tất cả hàm nhận DataFrame đã làm sạch (`clean_trips`), trả DataFrame Spark (trừ `airport_summary`).
`tip_rate` = `sum(tip_amount) / sum(fare_amount)`, làm tròn 4: dùng hàm `tip_rate()` viết sẵn.

| Hàm | Trả về |
| --- | --- |
| `peak_hours(df)` | cột `pickup_hour, trips`, sắp theo `pickup_hour` tăng dần |
| `tip_by_payment(df)` | cột `payment, trips, tip_rate, tipped_share`; `payment` là `"card"`/`"cash"`/`"other"` theo `PAYMENT`; `tipped_share` = tỉ lệ chuyến có `tip_amount > 0` (làm tròn 4); sắp `trips` giảm dần |
| `tip_rate_by_borough(trips, zones)` | cột `borough, trips, tip_rate`, **chỉ chuyến trả thẻ** (`payment_type == 1`), borough của **khu đón** (`pu_location_id`); sắp `tip_rate` giảm dần. `zones` có 265 dòng: **broadcast** nó, đừng để Spark shuffle 84 triệu dòng |
| `airport_summary(trips, zones)` | `dict` với `airport_trips` (chuyến đón **hoặc** trả ở zone có `service_zone` thuộc `AIRPORT_ZONES`), `airport_share` (làm tròn 4), `airport_avg_fare`, `other_avg_fare` (trung bình `fare_amount`, làm tròn 2) |

Vì sao `tip_rate_by_borough` chỉ lấy chuyến trả thẻ? Chạy `tip_by_payment` rồi nhìn dòng `cash`.

In [ ]:
def peak_hours(df: DataFrame) -> DataFrame:
    # TODO: groupBy pickup_hour, đếm thành cột trips, sắp theo giờ
    raise NotImplementedError


def tip_by_payment(df: DataFrame) -> DataFrame:
    # TODO: đổi payment_type thành nhãn theo PAYMENT (còn lại "other"), rồi tính trips, tip_rate, tipped_share
    raise NotImplementedError


def tip_rate_by_borough(trips: DataFrame, zones: DataFrame) -> DataFrame:
    # TODO: lọc chuyến thẻ, join F.broadcast(zones) theo pu_location_id == location_id, nhóm theo borough
    raise NotImplementedError


def airport_summary(trips: DataFrame, zones: DataFrame) -> dict:
    # TODO: lấy location_id các zone sân bay (bảng nhỏ, collect được), đánh dấu chuyến đón hoặc trả ở đó,
    #       rồi tính cả 4 con số trong MỘT lần agg (một lần quét dữ liệu)
    raise NotImplementedError

In [ ]:
# explore: đường cầu theo giờ, ngày thường vs cuối tuần (trung bình mỗi ngày)
hours = peak_hours(trips).toPandas().set_index("pickup_hour")["trips"]
PEAK_HOUR, PEAK_TRIPS = int(hours.idxmax()), int(hours.max())
print(f"giờ cao điểm {PEAK_HOUR}h: {PEAK_TRIPS:,} chuyến cả năm; giờ vắng nhất {int(hours.idxmin())}h: {int(hours.min()):,}")

by_kind = (
    trips.withColumn("weekend", F.dayofweek("pickup_date").isin(1, 7))
    .groupBy("weekend", "pickup_hour").agg(F.count("*").alias("trips"), F.countDistinct("pickup_date").alias("days"))
    .toPandas()
)
by_kind["per_day"] = by_kind["trips"] / by_kind["days"]
curve = by_kind.pivot(index="pickup_hour", columns="weekend", values="per_day").rename(
    columns={False: "ngày thường", True: "cuối tuần"})
print(curve.round(0).astype(int).loc[[0, 2, 5, 8, 18]])
ax = curve.plot(figsize=(10, 3.5), marker="o")
ax.set_title("Chuyến trung bình mỗi ngày theo giờ đón khách, Yellow Taxi NYC 2019")
ax.set_xlabel("giờ (giờ New York)")
ax.set_ylabel("chuyến / ngày")
plt.show()

In [ ]:
# explore: tip theo cách trả tiền và theo borough (chuyến thẻ)
payments = tip_by_payment(trips).toPandas()
print(payments)
t = time.perf_counter()
boroughs = tip_rate_by_borough(trips, zones).toPandas()
BROADCAST_SECONDS = time.perf_counter() - t
print(boroughs)
ax = boroughs.set_index("borough")["tip_rate"].sort_values().plot.barh(figsize=(8, 3.5))
ax.set_title("Tip / cước, chuyến trả thẻ, theo borough nơi đón khách (2019)")
ax.set_xlabel("tip_rate")
plt.show()

In [ ]:
# explore: sân bay, và các con số cho trắc nghiệm
AIRPORTS = airport_summary(trips, zones)
INSIGHTS = {
    "raw_trips": RAW_ROWS, "clean_trips": CLEAN_ROWS, "rejected_pct": round(REJECTED / RAW_ROWS * 100, 2),
    "peak_hour": PEAK_HOUR, "peak_trips": PEAK_TRIPS,
    "card_tip_rate": float(payments.set_index("payment").loc["card", "tip_rate"]),
    "cash_tipped_share": float(payments.set_index("payment").loc["cash", "tipped_share"]),
    **AIRPORTS,
}
INSIGHTS

## 5. Tối ưu: đo trước / sau

Ba cell dưới so sánh cùng một việc làm theo hai cách. Thời gian in ra chỉ để cảm nhận (phụ thuộc máy,
cache hệ điều hành); test chấm bằng **physical plan** và **số tệp**, không bằng đồng hồ.

1. **Broadcast join**: tắt broadcast tự động (`autoBroadcastJoinThreshold = -1`) và bỏ hint thì Spark
   chọn `SortMergeJoin`: shuffle cả 60 triệu chuyến thẻ để join với 265 dòng.
2. **Small files**: ghi tháng 1 sau `repartition(64)` mà không repartition theo ngày.
3. **Partition pruning**: đếm chuyến ngày 04/07 từ tệp gốc (quét cả năm) vs từ dữ liệu đã partition (đọc một thư mục).

In [ ]:
# explore: (1) broadcast hash join vs sort-merge join trên cùng phép tính
def join_name(df):
    plan = df._jdf.queryExecution().executedPlan().toString()
    return next(n for n in ("BroadcastHashJoin", "SortMergeJoin", "ShuffledHashJoin") if n in plan)


key = "spark.sql.autoBroadcastJoinThreshold"
old = spark.conf.get(key)
spark.conf.set(key, "-1")
try:
    card = trips.where(F.col("payment_type") == 1)
    smj = (card.join(zones, card["pu_location_id"] == zones["location_id"])     # không hint
           .groupBy("borough").agg(F.count("*").alias("trips"), tip_rate()))
    t = time.perf_counter()
    smj.collect()
    smj_seconds = time.perf_counter() - t
    print(f"không hint, tắt auto-broadcast: {join_name(smj)}, {smj_seconds:.1f}s")
    bhj = tip_rate_by_borough(trips, zones)
    t = time.perf_counter()
    bhj.collect()
    print(f"tip_rate_by_borough (hint broadcast): {join_name(bhj)}, {time.perf_counter() - t:.1f}s")
finally:
    spark.conf.set(key, old)

In [ ]:
# explore: (2) small files: cùng dữ liệu tháng 1, hai cách ghi
import shutil

jan = trips.where(F.col("pickup_date") < "2019-02-01").repartition(64)   # rải đều 64 partition như sau shuffle
scratch = Path(tempfile.mkdtemp(prefix="nyc-small-files-"))
t = time.perf_counter()
jan.write.mode("overwrite").partitionBy("pickup_date").parquet(str(scratch / "naive"))
naive_seconds = time.perf_counter() - t
t = time.perf_counter()
write_clean(jan, scratch / "fixed")
fixed_seconds = time.perf_counter() - t
for name, seconds in (("naive", naive_seconds), ("fixed", fixed_seconds)):
    n_files = len(list((scratch / name).glob("pickup_date=*/*.parquet")))
    print(f"{name:>5}: {n_files:5d} tệp cho 31 ngày, ghi {seconds:.1f}s")
t = time.perf_counter()
spark.read.parquet(str(scratch / "naive")).agg(F.sum("fare_amount")).collect()
print(f"đọc lại bản naive: {time.perf_counter() - t:.1f}s")
t = time.perf_counter()
spark.read.parquet(str(scratch / "fixed")).agg(F.sum("fare_amount")).collect()
print(f"đọc lại bản fixed: {time.perf_counter() - t:.1f}s")
shutil.rmtree(scratch)

In [ ]:
# explore: (3) partition pruning: ngày 04/07/2019
import re

t = time.perf_counter()
from_raw = clean_trips(raw).where(F.col("pickup_date") == "2019-07-04").count()
raw_seconds = time.perf_counter() - t
day = spark.read.parquet(str(CLEAN)).where(F.col("pickup_date") == "2019-07-04")
t = time.perf_counter()
from_clean = day.count()
print(f"từ tệp gốc: {from_raw:,} chuyến, {raw_seconds:.1f}s (quét 12 tháng)")
print(f"từ dữ liệu partition: {from_clean:,} chuyến, {time.perf_counter() - t:.2f}s")
plan = day._jdf.queryExecution().executedPlan().toString()
print(re.search(r"PartitionFilters: \[[^\]]*\]", plan).group(0))   # Spark chỉ mở 1/365 thư mục

## 6. Báo cáo bài học

Làm phần trắc nghiệm trong app sau khi chạy notebook.

In [ ]:
# explore: trả tài nguyên
spark.stop()